# Inspect Last Frames Across Splits

Visual QA notebook for a generated dataset.

1. Set `dataset_root` to a folder under `datasets/`.
2. Run all cells.
3. The notebook scans `train`, `valid`, and `test`, then displays the **last time frame** for sampled trajectories.

In [ ]:
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np

try:
    import torch
except Exception:
    torch = None

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 110

In [ ]:
def to_numpy(x: Any) -> np.ndarray:  # noqa: D103
    if isinstance(x, np.ndarray):
        return x
    if torch is not None and isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def load_split_data(split_dir: Path) -> np.ndarray:  # noqa: D103, PLR0912
    candidates = [split_dir / "data.pt", split_dir / "data.npz", split_dir / "data.npy"]
    existing = next((p for p in candidates if p.exists()), None)
    if existing is None:
        raise FileNotFoundError(f"No supported data file found in {split_dir}")

    if existing.suffix == ".pt":
        if torch is None:
            msg = "PyTorch is required to read .pt files"
            raise ImportError(msg)
        obj = torch.load(existing, map_location="cpu")
    elif existing.suffix == ".npz":
        obj = np.load(existing, allow_pickle=True)
    else:
        obj = np.load(existing, allow_pickle=True)

    if isinstance(obj, dict):
        if "data" in obj:
            return to_numpy(obj["data"])
        for value in obj.values():
            arr = to_numpy(value)
            if arr.ndim >= 3:
                return arr
        raise ValueError(f"No array-like payload found in dict from {existing}")

    if isinstance(obj, np.lib.npyio.NpzFile):
        if "data" in obj.files:
            return to_numpy(obj["data"])
        for key in obj.files:
            arr = to_numpy(obj[key])
            if arr.ndim >= 3:
                return arr
        raise ValueError(f"No array-like payload found in npz from {existing}")

    return to_numpy(obj)


def canonicalize_to_nthwc(arr: np.ndarray) -> np.ndarray:  # noqa: D103
    # Target layout: (N, T, H, W, C)
    arr = np.asarray(arr)

    if arr.ndim == 3:  # (N, H, W)
        arr = arr[:, None, :, :, None]
    elif arr.ndim == 4:
        # Ambiguous 4D data. Assume either (N, T, H, W) or (N, H, W, C).
        if arr.shape[1] > 8 and arr.shape[-1] <= 8:
            arr = arr[:, None, :, :, :]
        else:
            arr = arr[:, :, :, :, None]
    elif arr.ndim == 5:
        # Move likely time axis to index 1.
        time_axis = 1 + int(np.argmax(arr.shape[1:]))
        arr = np.moveaxis(arr, time_axis, 1)

        # Of remaining 3 axes, smallest is usually channel axis.
        spatial_and_channel = arr.shape[2:]
        channel_axis = 2 + int(np.argmin(spatial_and_channel))
        arr = np.moveaxis(arr, channel_axis, -1)
    else:
        raise ValueError(f"Unsupported data ndim={arr.ndim}; expected 3-5 dims")

    if arr.shape[-1] == 0:
        msg = "Channel dimension inferred as zero"
        raise ValueError(msg)

    return arr

In [ ]:
# Edit this relative path to whichever dataset you want to inspect.
# dataset_rel_path = "gpe_ri_low_complexity_89373b1"
dataset_rel_path = "2026-03-26/conditioned_navier_stokes_2d_b106e4d"
# dataset_rel_path = "2026-03-27/gpe/laser_only_wake_8419c8a"
dataset_rel_path = "2026-03-26/shallow_water2d_433068d"
# dataset_rel_path = "2026-03-27/gpe/laser_only_wake_65c1425"
# dataset_rel_path = "2026-03-28/gpe/laser_only_wake_d0ca1bf"
dataset_rel_path = "2026-03-30/gpe/laser_only_wake_5b51eac"
# dataset_rel_path = "2026-03-30/gpe/laser_only_wake_f15bdbb"

candidate_roots = [Path("datasets"), Path("../datasets"), Path("../outputs")]
dataset_root = None
for base in candidate_roots:
    maybe = (base / dataset_rel_path).resolve()
    if maybe.exists():
        dataset_root = maybe
        break

if dataset_root is None:
    dataset_root = (candidate_roots[0] / dataset_rel_path).resolve()

# First sample index to display.
start_idx = 100

# Number of trajectories to show from each split.
num_samples_per_split = 100

# Maximum channels to show per trajectory.
max_channels_to_show = 3

print("Dataset root:", dataset_root)
print("Start index:", start_idx)

In [ ]:
def show_last_frames_for_split(split_name: str) -> None:  # noqa: D103
    split_dir = dataset_root / split_name # type: ignore  # noqa: PGH003
    if not split_dir.exists():
        print(f"[{split_name}] not found, skipping")
        return

    raw = load_split_data(split_dir)
    data = canonicalize_to_nthwc(raw)

    n, t, h, w, c = data.shape
    if start_idx < 0:
        msg = "start_idx must be >= 0"
        raise ValueError(msg)

    available = max(0, n - start_idx)
    rows = min(num_samples_per_split, available)
    cols = min(max_channels_to_show, c)

    if rows == 0:
        print(
            f"[{split_name}] shape={data.shape}  no samples to show for "
            f"start_idx={start_idx}"
        )
        return

    print(
        f"[{split_name}] shape={data.shape}  "
        f"showing idx={start_idx}..{start_idx + rows - 1}, "
        f"channels={cols}, last_t={t - 1}"
    )

    fig, axes = plt.subplots(
        rows, cols, figsize=(3.8 * cols, 3.2 * rows), squeeze=False
    )

    for i in range(rows):
        sample_idx = start_idx + i
        last_frame = data[sample_idx, -1]
        for ch in range(cols):
            ax = axes[i, ch]
            im = ax.imshow(last_frame[..., ch], cmap="viridis")
            ax.set_title(f"{split_name} sample {sample_idx} ch {ch}")
            ax.set_xticks([])
            ax.set_yticks([])
            fig.colorbar(im, ax=ax, fraction=0.045, pad=0.02)

    fig.tight_layout()
    plt.show()


for split in ["train", "valid", "test"]:
    show_last_frames_for_split(split)